In [1]:
import subprocess, sys
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True)
print(result.stdout.strip())
import torch
assert torch.cuda.is_available(), "No GPU detected"
props = torch.cuda.get_device_properties(0)
assert props.total_memory > 30e9, (
    f"Need ≥40 GB VRAM. Got {props.total_memory/1e9:.0f} GB. Switch to A100.")
print(f"GPU: {props.name} — {props.total_memory/1e9:.0f} GB")

NVIDIA A100-SXM4-40GB, 40960
GPU: NVIDIA A100-SXM4-40GB — 42 GB


In [2]:
!pip install flask pyngrok nilearn nibabel -q

In [3]:
import subprocess, sys

# 1. Pin NumPy
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.26.4,<2.1.0'])

# 2. Fix torchvision to match torch 2.10.0+cu128
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torchvision==0.25.0',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu124',
                '--no-deps'])  # --no-deps prevents it dragging in wrong torch

# 3. Install tribev2
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git'])

print("✓ All done — NOW do Runtime → Restart Runtime, then verify below")

✓ All done — NOW do Runtime → Restart Runtime, then verify below


In [4]:
import os
# Load token from Colab Secrets
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    from huggingface_hub import login
    login()

HF_TOKEN loaded from Colab Secrets


In [5]:
from pathlib import Path
from tribev2.demo_utils import TribeModel
import torch
CACHE_DIR = Path('/content/tribe_cache')
CACHE_DIR.mkdir(exist_ok=True)
print('Loading TRIBE v2 (first run downloads ~1 GB)...')
model = TribeModel.from_pretrained(
    'facebook/tribev2',
    cache_folder=str(CACHE_DIR)
)
print('Model loaded')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM after load: {used:.1f} / {total:.1f} GB')

/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-04-26 12:28:46 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


Loading TRIBE v2 (first run downloads ~1 GB)...


config.yaml: 0.00B [00:00, ?B/s]

best.ckpt:   0%|          | 0.00/709M [00:00<?, ?B/s]

2026-04-26 12:28:51 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
INFO:tribev2.demo_utils:Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use 

Model loaded
VRAM after load: 0.7 / 42.4 GB


In [6]:
import numpy as np
from nilearn import datasets as nl_datasets
from nilearn.plotting import view_surf
from IPython.display import display, HTML
N_PER_HEMI = 10242   # fsaverage5: 10242 vertices per hemisphere
print('Fetching fsaverage5 mesh...')
fsavg = nl_datasets.fetch_surf_fsaverage(mesh='fsaverage5')
print('Mesh ready')
print('Keys:', [k for k in fsavg.keys() if k != 'description'])

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x791991573ce0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


Fetching fsaverage5 mesh...
Mesh ready
Keys: ['area_left', 'area_right', 'curv_left', 'curv_right', 'flat_left', 'flat_right', 'infl_left', 'infl_right', 'pial_left', 'pial_right', 'sphere_left', 'sphere_right', 'sulc_left', 'sulc_right', 'thick_left', 'thick_right', 'white_left', 'white_right']


In [7]:
import numpy as np
import nibabel as nib
import urllib.request, os
from nilearn.surface import load_surf_mesh

os.makedirs('/content/tribe_cache', exist_ok=True)
print("Downloading HCP-MMP1 atlas...")
atlas_path = '/content/tribe_cache/MMP_in_MNI_corr.nii.gz'
if not os.path.exists(atlas_path):
    urllib.request.urlretrieve(
        'https://ndownloader.figshare.com/files/5594363', atlas_path)

atlas_img  = nib.load(atlas_path)
atlas_data = np.asarray(atlas_img.get_fdata(), dtype=np.int32)
inv_affine = np.linalg.inv(atlas_img.affine)

lh_coords  = load_surf_mesh(fsavg['pial_left'])[0]
rh_coords  = load_surf_mesh(fsavg['pial_right'])[0]
all_coords = np.vstack([lh_coords, rh_coords])

coords_h = np.hstack([all_coords, np.ones((len(all_coords), 1))])
vox_ijk  = np.round((inv_affine @ coords_h.T).T[:, :3]).astype(int)
nx, ny, nz = atlas_data.shape
valid = (
    (vox_ijk[:,0] >= 0) & (vox_ijk[:,0] < nx) &
    (vox_ijk[:,1] >= 0) & (vox_ijk[:,1] < ny) &
    (vox_ijk[:,2] >= 0) & (vox_ijk[:,2] < nz)
)
vertex_labels = np.zeros(len(all_coords), dtype=np.int32)
vi = vox_ijk[valid]
vertex_labels[valid] = atlas_data[vi[:,0], vi[:,1], vi[:,2]]

GLASSER = {
    "V4": 6, "FFC": 18, "MT": 23, "MST": 2, "V4t": 156, "FST": 157,
    "a24": 61, "p24": 179, "p32": 63, "s32": 64, "24dd": 40, "24dv": 41,
    "46": 84, "9-46d": 86, "p9-46v": 83, "a9-46v": 85,
    "AAIC": 112, "AVI": 111, "MI": 109, "PI": 110,
}
REGION_MAP = {
    "FFA":    ("FFC",),
    "V4":     ("V4",),
    "MT+":    ("MT", "MST", "V4t", "FST"),
    "PFC":    ("46", "9-46d", "p9-46v", "a9-46v"),
    "ACC":    ("a24", "p24", "p32", "s32", "24dd", "24dv"),
    "Insula": ("AAIC", "AVI", "MI", "PI"),
}

region_masks = {}
for region, names in REGION_MAP.items():
    idx = [i for n in names for lh in [GLASSER.get(n)] if lh for i in (lh, lh+180)]
    if idx:
        region_masks[region] = np.isin(vertex_labels, idx)
        print(f"  {region:8s}: {int(region_masks[region].sum())} vertices")

  FFA     : 25 vertices
  V4      : 35 vertices
  MT+     : 172 vertices
  PFC     : 310 vertices
  ACC     : 360 vertices
  Insula  : 264 vertices


In [8]:
import nibabel as nib
import numpy as np
from nilearn import datasets as nl_datasets

# fetch_atlas_harvard_oxford returns the image directly in newer nilearn
atlas = nl_datasets.fetch_atlas_harvard_oxford('sub-maxprob-thr25-2mm')

# Handle both old and new nilearn versions
if hasattr(atlas.maps, 'get_fdata'):
    # Newer nilearn — atlas.maps is already a Nifti1Image
    sub_img  = atlas.maps
else:
    # Older nilearn — atlas.maps is a file path
    sub_img  = nib.load(atlas.maps)

sub_data   = np.asarray(sub_img.get_fdata(), dtype=np.int32)
sub_affine = sub_img.affine

SUBCORTICAL_LABELS = {
    'Hippocampus': [6, 13],
    'Amygdala':    [7, 14],
    'NAcc':        [8, 15],
}

subcortical_masks = {}
for region, label_ids in SUBCORTICAL_LABELS.items():
    mask       = np.isin(sub_data, label_ids)
    vox_coords = np.argwhere(mask)
    mni_coords = nib.affines.apply_affine(sub_affine, vox_coords)
    subcortical_masks[region] = mni_coords
    print(f"  {region:15s}: {len(mni_coords)} voxels")

print("✓ Subcortical masks ready")

[fetch_atlas_harvard_oxford] Added README.md to /root/nilearn_data

[fetch_atlas_harvard_oxford] Dataset created in /root/nilearn_data/fsl

[fetch_atlas_harvard_oxford] Downloading data from https://www.nitrc.org/frs/download.php/9902/HarvardOxford.tgz 
...

[fetch_atlas_harvard_oxford] Downloaded 385024 of 25716861 bytes (1.5%%,  1.1min remaining)

[fetch_atlas_harvard_oxford] Downloaded 9011200 of 25716861 bytes (35.0%%,    4.0s remaining)

[fetch_atlas_harvard_oxford] Downloaded 23511040 of 25716861 bytes (91.4%%,    0.3s remaining)

[fetch_atlas_harvard_oxford]  ...done. (4 seconds, 0 min)

[fetch_atlas_harvard_oxford] Extracting data from 
/root/nilearn_data/fsl/a53efddf76ebce4e6927de851d9126a0/HarvardOxford.tgz...

[fetch_atlas_harvard_oxford] .. done.

  Hippocampus    : 64853 voxels
  Amygdala       : 1286 voxels
  NAcc           : 6173 voxels
✓ Subcortical masks ready


In [9]:
import cv2, io, os, tempfile
import numpy as np
import pandas as pd
from PIL import Image
from scipy.spatial import cKDTree
from nilearn.surface import load_surf_mesh
from tribev2.demo_utils import get_audio_and_text_events

def tribe_predict_image(img_bytes: bytes) -> dict:
    img    = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img_np = np.array(img)
    h, w   = img_np.shape[:2]
    w, h   = w - w % 2, h - h % 2
    img_np = img_np[:h, :w]

    with tempfile.TemporaryDirectory() as tmpdir:
        video_path = os.path.join(tmpdir, 'ui.mp4')
        writer = cv2.VideoWriter(
            video_path, cv2.VideoWriter_fourcc(*'mp4v'), 1, (w, h))
        for _ in range(2):
            writer.write(cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
        writer.release()

        event_df = pd.DataFrame([{
            "type":     "Video",
            "filepath": video_path,
            "start":    0,
            "timeline": "default",
            "subject":  "default",
        }])
        events   = get_audio_and_text_events(event_df)
        preds, _ = model.predict(events=events)

    preds_np = np.asarray(preds)
    mean_act = preds_np.mean(axis=0)

    # Cortical region scores
    scores = {
        r: float(mean_act[m].mean()) if m.sum() > 0 else 0.0
        for r, m in region_masks.items()
    }
    lo, hi = min(scores.values()), max(scores.values())
    if hi - lo > 1e-6:
        scores = {k: (v - lo) / (hi - lo) for k, v in scores.items()}

    # Subcortical estimates via nearest cortical vertex
    lh_coords  = load_surf_mesh(fsavg['pial_left'])[0]
    rh_coords  = load_surf_mesh(fsavg['pial_right'])[0]
    all_coords = np.vstack([lh_coords, rh_coords])
    tree       = cKDTree(all_coords)

    subcortical_scores = {}
    for region, mni_coords in subcortical_masks.items():
        _, idx = tree.query(mni_coords)
        subcortical_scores[region] = float(mean_act[idx].mean())

    lo2, hi2 = min(subcortical_scores.values()), max(subcortical_scores.values())
    if hi2 - lo2 > 1e-6:
        subcortical_scores = {k: (v - lo2) / (hi2 - lo2)
                              for k, v in subcortical_scores.items()}

    return {
        "vertices": {
            "lh": mean_act[:10242].tolist(),
            "rh": mean_act[10242:].tolist(),
        },
        "region_scores": scores,
        "subcortical_estimates": {
            "values": subcortical_scores,
            "method": "nearest_cortical_vertex",
            "warning": "Cortical proxy estimates only. TRIBE v2 models cortical activity only."
        },
        "meta": {
            "n_vertices_per_hemi": 10242,
            "n_timesteps": int(preds_np.shape[0]),
            "surface": "fsaverage5",
        }
    }

In [10]:
checks = {
    'model':               'model' in globals(),
    'region_masks':        'region_masks' in globals(),
    'subcortical_masks':   'subcortical_masks' in globals(),
    'tribe_predict_image': 'tribe_predict_image' in globals(),
    'fsavg':               'fsavg' in globals(),
}
for name, ok in checks.items():
    print(f"{'✓' if ok else '✗'} {name}")
assert all(checks.values()), "Fix ✗ items before starting server"
print("\n✓ All good — safe to start server")

✓ model
✓ region_masks
✓ subcortical_masks
✓ tribe_predict_image
✓ fsavg

✓ All good — safe to start server


In [11]:
import subprocess, torch

# Check GPU memory
free_gpu, total_gpu = torch.cuda.mem_get_info()
print(f"GPU: {free_gpu/1e9:.1f} GB free / {total_gpu/1e9:.1f} GB total")

# Check system RAM
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(result.stdout)

# Check if anything is already using the GPU
result2 = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result2.stdout)

import subprocess
result = subprocess.run(['fuser', '8080/tcp'], capture_output=True, text=True)
print("Port 8080 in use by PID:", result.stdout.strip() or "nobody")

# Also check flask is installed
import flask
print("Flask version:", flask.__version__)

GPU: 41.2 GB free / 42.4 GB total
               total        used        free      shared  buff/cache   available
Mem:            83Gi       2.6Gi        35Gi        16Mi        45Gi        80Gi
Swap:             0B          0B          0B

Sun Apr 26 12:29:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |              

/tmp/ipykernel_7956/3968297838.py:21: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:", flask.__version__)


In [12]:
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok
import os
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

srv = Flask(__name__)

@srv.route('/encode', methods=['POST'])
def encode():
    if not request.data:
        return jsonify({'error': 'POST raw PNG bytes as request body'}), 400
    return jsonify(tribe_predict_image(request.data))

@srv.route('/health')
def health():
    return jsonify({'ok': True})

PORT = 5000
os.system(f"fuser -k {PORT}/tcp")
threading.Thread(target=lambda: srv.run(port=PORT, use_reloader=False), daemon=True).start()
tunnel = ngrok.connect(PORT)
print("TRIBE endpoint:", tunnel.public_url + "/encode")
print("Health check: ", tunnel.public_url + "/health")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


TRIBE endpoint: https://deluge-glucose-rework.ngrok-free.dev/encode
Health check:  https://deluge-glucose-rework.ngrok-free.dev/health
